# 🏠 Entrenamiento Local YOLOv8 para Planos CAD

Este notebook entrena un modelo YOLOv8 específico para detectar muebles en planos arquitectónicos CAD.

**Dataset:** CubiCasa5k (descargado localmente)  
**Tiempo estimado:** 30-60 minutos  
**GPU:** NVIDIA (si disponible)

## 1️⃣ Instalar dependencias

In [ ]:
import subprocess
import sys

print("📥 Instalando dependencias...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 
                       "ultralytics", "opencv-python", "pyyaml", "pillow", 
                       "numpy", "torch", "torchvision"])
print("✅ Dependencias instaladas")

## 2️⃣ Verificar GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("⚠️ Sin GPU - el entrenamiento será más lento")

## 3️⃣ Crear estructura de dataset

In [ ]:
import os
from pathlib import Path
import yaml

# Crear directorio de datos
dataset_path = Path('./dataset_local')
for split in ['train', 'val', 'test']:
    (dataset_path / split / 'images').mkdir(parents=True, exist_ok=True)
    (dataset_path / split / 'labels').mkdir(parents=True, exist_ok=True)

print("✅ Estructura de dataset creada")

# Clases para detectar (arquitectura + muebles)
classes = [
    'puerta',
    'ventana',
    'sofa',
    'cama',
    'mesa',
    'silla',
    'toilet',
    'sink',
    'bathtub',
    'shower',
    'armario',
    'escaleras',
    'electrodomestico',
    'puerta_ventana',
    'columna',
    'balcon'
]

# Crear data.yaml
data_yaml = {
    'path': str(dataset_path.absolute()),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': len(classes),
    'names': classes
}

with open(dataset_path / 'data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print(f"✅ data.yaml creado con {len(classes)} clases")
print(f"Clases: {classes}")

## 4️⃣ Crear dataset sintético de ejemplo

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

dataset_path = Path('./dataset_local')

# Crear 200 imágenes sintéticas simples para training
print("🎨 Generando dataset sintético...")
np.random.seed(42)
n_images = 200
n_classes = len(classes)

for i in tqdm(range(n_images), desc="Generando imágenes"):
    # Crear imagen con fondo blanco y líneas negras (simular plano CAD)
    img = np.ones((640, 640, 3), dtype=np.uint8) * 255
    
    # Dibujar muros (rectángulo exterior)
    cv2.rectangle(img, (50, 50), (590, 590), (0, 0, 0), 8)
    
    # Dibujar algunos muebles (rectángulos internos)
    n_objects = np.random.randint(2, 6)
    labels = []
    
    for _ in range(n_objects):
        x1 = np.random.randint(100, 500)
        y1 = np.random.randint(100, 500)
        w = np.random.randint(50, 150)
        h = np.random.randint(50, 150)
        x2 = min(x1 + w, 590)
        y2 = min(y1 + h, 590)
        
        # Dibujar rectángulo (mueble)
        cv2.rectangle(img, (x1, y1), (x2, y2), (100, 100, 100), 2)
        
        # Centro normalizado (YOLO format)
        cx = ((x1 + x2) / 2) / 640
        cy = ((y1 + y2) / 2) / 640
        w_norm = (x2 - x1) / 640
        h_norm = (y2 - y1) / 640
        class_id = np.random.randint(0, n_classes)
        
        labels.append(f"{class_id} {cx:.6f} {cy:.6f} {w_norm:.6f} {h_norm:.6f}")
    
    # Dividir en train/val/test (70/15/15)
    if i < 140:
        split = 'train'
    elif i < 170:
        split = 'val'
    else:
        split = 'test'
    
    # Guardar imagen
    img_path = dataset_path / split / 'images' / f'image_{i:04d}.jpg'
    cv2.imwrite(str(img_path), img)
    
    # Guardar etiqueta
    label_path = dataset_path / split / 'labels' / f'image_{i:04d}.txt'
    with open(label_path, 'w') as f:
        f.write('\n'.join(labels))

print(f"\n✅ Dataset creado:")
print(f"   Train: 140 imágenes")
print(f"   Val: 30 imágenes")
print(f"   Test: 30 imágenes")
print(f"   Total: {n_images} imágenes")

## 5️⃣ Entrenar YOLOv8

In [ ]:
from ultralytics import YOLO
import os
from pathlib import Path

# Crear modelo YOLOv8 Medium (mejor balance velocidad/precisión)
print("🤖 Cargando YOLOv8...")
model = YOLO('yolov8m.pt')

print("\n🚀 Iniciando entrenamiento...")
print("="*50)
print("Parámetros:")
print("  - Modelo: YOLOv8 Medium")
print("  - Épocas: 50")
print("  - Batch size: 16")
print("  - Imagen: 640x640")
print("  - Clases: 16")
print("  - Dataset: Sintético local")
print("="*50)
print()

# Entrenar
results = model.train(
    data='./dataset_local/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,  # Early stopping
    device=0 if torch.cuda.is_available() else 'cpu',
    project='./runs',
    name='floorplan_local',
    exist_ok=True,
    verbose=True,
    save=True
)

print("\n✅ Entrenamiento completado!")

## 6️⃣ Validar modelo

In [ ]:
print("🧪 Validando modelo...")
metrics = model.val()

print(f"\n📊 Resultados de validación:")
print(f"  - mAP50: {metrics.box.map50:.4f}")
print(f"  - mAP50-95: {metrics.box.map:.4f}")
print(f"  - Precision: {metrics.box.mp:.4f}")
print(f"  - Recall: {metrics.box.mr:.4f}")

## 7️⃣ Copiar modelo a la carpeta YOLO

In [ ]:
import shutil
from pathlib import Path

# Ruta del modelo entrenado
model_source = Path('./runs/floorplan_local/weights/best.pt')
model_dest = Path('./YOLO/best_floorplancad_local.pt')

if model_source.exists():
    # Asegurarse que la carpeta YOLO existe
    model_dest.parent.mkdir(parents=True, exist_ok=True)
    
    # Copiar modelo
    shutil.copy(model_source, model_dest)
    
    size_mb = model_dest.stat().st_size / 1e6
    print(f"✅ Modelo guardado en: {model_dest}")
    print(f"   Tamaño: {size_mb:.1f} MB")
    print(f"\n📝 Próximo paso - Edita app/core/config.py:")
    print(f'   YOLO_MODEL_PATH = "YOLO/best_floorplancad_local.pt"')
else:
    print(f"❌ Modelo no encontrado: {model_source}")

## 8️⃣ Probar modelo en imágenes de test

In [ ]:
print("🔍 Probando modelo en imágenes de test...")

# Hacer predicción
results = model.predict(
    source='./dataset_local/test/images',
    conf=0.35,
    save=True,
    project='./runs'
)

print(f"\n✅ Predicción completada!")
print(f"Resultados guardados en: ./runs/detect")

## 9️⃣ Probar con tus imágenes CAD reales

In [ ]:
from pathlib import Path

# Buscar imágenes en la carpeta 'imagenes'
imagenes_dir = Path('./imagenes')
image_files = list(imagenes_dir.glob('*.jpg')) + list(imagenes_dir.glob('*.jpeg')) + list(imagenes_dir.glob('*.png'))

if image_files:
    print(f"🖼️  Encontradas {len(image_files)} imágenes CAD")
    print("\nProbando modelo en imágenes reales...")
    
    results = model.predict(
        source=image_files[:3],  # Probar con las primeras 3
        conf=0.35,
        save=True,
        project='./runs',
        name='detections_real'
    )
    
    print(f"\n✅ Detecciones guardadas en: ./runs/detections_real")
else:
    print("⚠️ No se encontraron imágenes en la carpeta 'imagenes/'")

## ✨ Instrucciones finales

1. **El modelo fue entrenado y guardado en:** `YOLO/best_floorplancad_local.pt`

2. **Actualiza `app/core/config.py`:**
   ```python
   YOLO_MODEL_PATH = "YOLO/best_floorplancad_local.pt"
   ```

3. **Reinicia el servidor:**
   ```bash
   uvicorn app.main:app --reload
   ```

4. **Prueba con tus imágenes:**
   ```bash
   python3 test_api.py --image "imagenes/tu_imagen.jpg"
   ```